In [ ]:
import json
import os

base_path = os.path.abspath(os.path.dirname(__file__)) if '__file__' in globals() else os.getcwd()
json_path = os.path.join(base_path, 'COBI_Semantikmodell_02042025.json')

with open(json_path, 'r', encoding='utf-8') as f:
    json_config = json.load(f)
    


In [ ]:
import os
import re
import json
import urllib.parse
from datetime import datetime
from decimal import Decimal
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer, util
import chromadb
from sqlalchemy import create_engine, text
import torch
import ast

class SQLRAGPromptGenerator:
    _instance = None

    @staticmethod
    def get_instance():
        if SQLRAGPromptGenerator._instance is None:
            SQLRAGPromptGenerator()
        return SQLRAGPromptGenerator._instance

    def __init__(self):
        if SQLRAGPromptGenerator._instance is not None:
            raise Exception("This class is a singleton!")
        else:
            SQLRAGPromptGenerator._instance = self

        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        #self.rag_collection = self.chroma_client.get_collection("COBI_Knowledge_RAG")
        #sql_queries_enriched
        self.rag_collection = self.chroma_client.get_collection("sql_queries_enriched")
        self.wiki_collection = self.chroma_client.get_collection("COBI_Wiki_Mapping")

        self.alias_mapping = {
            "total_rest": "SUM(rest)",
            "total_RestGesamtgewicht": "SUM(RestGesamtgewicht)",
            "total_verbrauch": "SUM(verbrauch)",
            "total_lagerwert": "SUM(restlagerwert)",
            "total_vpehgeliefert": "SUM(vpehgeliefert)"
        }

        self.field_condition_preference = {
            "lagerverbraucht": "IS NULL",
            "reserviertvon": "IS NOT NULL",
        }
        self.config = json_config
        
    def get_packaging_type(self, artikel_number: str):
        conn_str = "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(
            "DRIVER={ODBC Driver 17 for SQL Server};"
            "SERVER=localhost;"
            "DATABASE=topwws;"
            "UID=sa;"
            "PWD=usersystemadmin!;"
            "TrustServerCertificate=yes"
        )
        engine = create_engine(conn_str)
        try:
            with engine.connect() as conn:
                #result = conn.execute(text("""
                    #SELECT
                        #COUNT(CASE WHEN colligeliefert IS NOT NULL THEN 1 END) AS anzahl_fixgebinde,
                        #COUNT(CASE WHEN colligeliefert = '' THEN 1 END) AS anzahl_gewichtsvariable
                    #FROM lagerliste
                    #WHERE artikelnr = :artikel
               # """), {"artikel": artikel_number}).fetchone()
                sql = text("""
                SELECT
                    CASE 
                        WHEN colligeliefert IS NOT NULL THEN 1 
                        ELSE 0 
                    END AS anzahl_fixgebinde,
                    CASE 
                        WHEN colligeliefert IS NULL THEN 1 
                        ELSE 0 
                    END AS anzahl_gewichtsvariable
                FROM (
                    SELECT TOP 1 colligeliefert
                    FROM lagerliste
                    WHERE artikelnr = :artikel
                    ORDER BY lagerzugang DESC
                ) AS latest_row;
            """)
                result = conn.execute(sql, {"artikel": artikel_number}).fetchone()
                fixgebinde = result.anzahl_fixgebinde
                gewichtsvariable = result.anzahl_gewichtsvariable
    
                if fixgebinde > 0:
                    return "fixgebinde"
                elif gewichtsvariable > 0:
                    return "Gewichtsartikel"
                else:
                    return "unknown"
    
        except Exception as e:
            print(f"❌ Packaging type check failed: {e}")
            return "unknown"
    '''        
    def determine_and_cache_packaging(self, user_query):
        match = re.search(r"(?:artikel|artikelnr)[^\d]*(\d{5,})", user_query.lower())
        if not match:
            print(" No artikelnr found in user query.")
            self.packaging_type = "unknown"
        else:
            artikel_number = match.group(1)
            self.packaging_type = self.get_packaging_type(artikel_number)
        print(" Cached packaging type:", self.packaging_type)
    	return self.packaging_type
    '''
    def determine_and_cache_packaging(self, user_query):
        match = re.search(r"(?:artikel|artikelnr)[^\d]*(\d{5,})", user_query.lower())
        if not match:
            print(" No artikelnr found in user query.")
            self.packaging_type = "unknown"
        else:
            artikel_number = match.group(1)
            self.packaging_type = self.get_packaging_type(artikel_number)
        print(" Cached packaging type:", self.packaging_type)
        return self.packaging_type


    def apply_field_aliases(self, fields: list[str]) -> list[str]:
        """
        Convert fields using alias_mapping.
        E.g., 'vpehgeliefert' becomes 'SUM(vpehgeliefert) AS total_vpehgeliefert'
        if defined in alias_mapping.
        """
        resolved_fields = []
    
        for field in fields:
            found_alias = False
    
            for alias_name, expression in self.alias_mapping.items():
                # Check if this expression ends with the current field
                if expression.endswith(f"({field})") or expression == f"SUM({field})":
                    resolved_fields.append(f"{expression} AS {alias_name}")
                    found_alias = True
                    break
    
            if not found_alias:
                resolved_fields.append(field)

        return resolved_fields

    def get_best_matching_wiki_entry(self, user_query, threshold=0.90):
        query_embedding = self.embedding_model.encode(user_query, convert_to_tensor=True).tolist()
        wiki_results = self.rag_collection.query(query_embeddings=[query_embedding], n_results=1)
        docs = wiki_results.get("documents", [[]])[0]
        metas = wiki_results.get("metadatas", [[]])[0]
        
        if not docs or not metas:
            return None
    
        doc_text = docs[0]
        meta = metas[0]
        doc_embedding = self.embedding_model.encode(doc_text, convert_to_tensor=True)
    
        # Ensure both tensors are on the same device
        query_tensor = torch.tensor(query_embedding).to(doc_embedding.device)
        similarity = util.cos_sim(query_tensor, doc_embedding).item()
    
        if similarity >= threshold:
            return {
                "document_text": doc_text,
                "metadata": meta,
                "similarity": similarity
            }
    
        return None
    def get_semantic_context_text(self):
        """
        Convert semantic model from self.config into readable context for the LLM.
        """
        lines = []
        config = self.config
    
        for key, section in config.items():
            if isinstance(section, dict):
                description = section.get("beschreibung", "")
                felder = section.get("felder") or section.get("typische_felder", [])
                intent = section.get("templateIntent", [])
                erkennung = section.get("erkennbar_an", "")
    
                lines.append(f"{key}:")
                if description:
                    lines.append(f"  Beschreibung: {description}")
                if felder:
                    lines.append(f"  Felder: {', '.join(felder)}")
                if intent:
                    lines.append(f"  Intent: {', '.join(intent)}")
                if erkennung:
                    lines.append(f"  Erkennbar an: {erkennung}")
                lines.append("")  # Blank line for readability
    
        return "\n".join(lines)


    def clean_where_conditions(self, conditions, user_query):
        #print("Raw input conditions:", conditions)
    
        normalized = set()
        field_map = {}
        query_lower = user_query.lower()
    
        for cond in conditions:
            cond = cond.strip()
            if not cond or ("=" not in cond and "IS" not in cond and ">=" not in cond):
                continue
    
            cond_upper = cond.upper()
            field = None
    
            if " IS NOT NULL" in cond_upper:
                field = cond.split(" IS")[0].strip().lower()
            elif " IS NULL" in cond_upper:
                field = cond.split(" IS")[0].strip().lower()
            elif ">=" in cond_upper:
                field = cond.split(">=")[0].strip().lower()
            elif "=" in cond_upper:
                field = cond.split("=")[0].strip().lower()
    
            if not field:
                normalized.add(cond)
                continue
    
            # Check if there's already a condition for this field
            existing = field_map.get(field)
            preferred = self.field_condition_preference.get(field)
    
            # Heuristic: detect if user implies NULL/NOT NULL for this field
            field_mentioned_negative = f"keine {field}" in query_lower or f"kein {field}" in query_lower
            field_mentioned_positive = (
                f"haben {field}" in query_lower or
                f"eine {field}" in query_lower or
                f"ein {field}" in query_lower or
                f"haben eine {field}" in query_lower
            )
    
            if existing:
                # Resolve conflicts using heuristics
                if "IS NULL" in cond_upper and "IS NOT NULL" in existing.upper():
                    if field_mentioned_negative:
                        field_map[field] = cond
                    elif field_mentioned_positive:
                        field_map[field] = existing
                elif "IS NOT NULL" in cond_upper and "IS NULL" in existing.upper():
                    if field_mentioned_positive:
                        field_map[field] = cond
                    elif field_mentioned_negative:
                        field_map[field] = existing
                else:
                    # fallback to longer condition
                    field_map[field] = cond if len(cond) > len(existing) else existing
            else:
                field_map[field] = cond
    
        cleaned = set(field_map.values()).union(normalized)
        #print("Final cleaned conditions:", cleaned)
        return cleaned

    
    def extract_where_clause(self, sql):
        match = re.search(r"\bWHERE\b\s+(.*?)(?:\bGROUP BY\b|\bORDER BY\b|$)", sql, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip().rstrip(";")
        #print("⚠️ No WHERE clause matched in SQL:", repr(sql))
        return None


    
    def resolve_relevant_templates(self, user_query, top_k=5, threshold=0.7):
        #print(f"\n Resolving templates for query: {user_query}")
        query_embedding = self.embedding_model.encode(user_query, convert_to_tensor=True)
    
        # Query the ChromaDB collection
        results = self.rag_collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
        #print(" Raw RAG results:", json.dumps(results, indent=2))
    
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
    
        if not metas:
            print(" No metadata found in RAG results.")
        if not docs:
            print(" No documents found in RAG results.")
    
        selected_fields = set()
        selected_conditions = set()
        used_any = False
    
        for doc, meta in zip(docs, metas):
            #print("\n Document:", doc)
            #print(" Metadata:", meta)
    
            doc_embedding = self.embedding_model.encode(doc, convert_to_tensor=True)
            similarity = util.cos_sim(query_embedding, doc_embedding).item()
            #print(f" Similarity with user query: {similarity:.4f}")
    
            if similarity >= threshold:
                used_any = True
                fields = meta.get("required_fields", "")
                selected_fields.update(f.strip() for f in fields.split(",") if f.strip())
    
                generated_sql = meta.get("generated_sql", "")
                #print(" Using SQL:", generated_sql)
                #print(" Raw SQL from matched template:", repr(generated_sql))
                where_clause = self.extract_where_clause(generated_sql)
                #print(" Extracted WHERE:", where_clause)
    
                if where_clause:
                    for c in re.split(r"\s+AND\s+", where_clause, flags=re.IGNORECASE):
                        selected_conditions.add(c.strip())
    
        # Fallback if none passed threshold
        if not used_any:
            #print("\n⚠️ No templates met threshold. Using fallback.")
            for meta in metas:
                fields = meta.get("required_fields", "")
                selected_fields.update(f.strip() for f in fields.split(",") if f.strip())
    
                generated_sql = meta.get("generated_sql", "")
                #print(" Fallback SQL:", generated_sql)
    
                where_clause = self.extract_where_clause(generated_sql)
                #print(" Fallback WHERE:", where_clause)
    
                if where_clause:
                    individual_conditions = [c.strip() for c in re.split(r"\s+AND\s+", where_clause, flags=re.IGNORECASE)]
                    selected_conditions.update(individual_conditions)
    
        #print("\n Final selected fields:", selected_fields)
        #print(" Final selected conditions:", selected_conditions)
    
        return sorted(selected_fields), sorted(selected_conditions)


    def resolve_aliases(self, fields, conditions, user_query):
        resolved_fields = []
        resolved_conditions = []
        seen_fields = set()
        lowered = user_query.lower()
    
        # Step 1: Resolve aliases for fields
        for f in fields:
            if f == "letzte_entnahme":
                if any(kw in lowered for kw in ["zugang", "zuletzt", "entnommen"]):
                    final_field = "lagerzugang AS letzte_entnahme"
                elif any(kw in lowered for kw in ["verbrauch", "nicht bewegt", "bewegung"]):
                    final_field = "lagerverbraucht AS letzte_entnahme"
                else:
                    final_field = "lagerzugang AS letzte_entnahme"
            else:
                final_field = self.alias_mapping.get(f, f)
    
            base_name = final_field.split(" AS ")[-1] if " AS " in final_field else final_field
            if base_name not in seen_fields:
                resolved_fields.append(final_field)
                seen_fields.add(base_name)
    
        # Step 2: Replace letzte_entnahme in WHERE conditions
        for cond in conditions:
            if "letzte_entnahme" in cond:
                if any(kw in lowered for kw in ["zugang", "zuletzt", "entnommen"]):
                    cond = cond.replace("letzte_entnahme", "lagerzugang")
                elif any(kw in lowered for kw in ["verbrauch", "nicht bewegt", "bewegung"]):
                    cond = cond.replace("letzte_entnahme", "lagerverbraucht")
                else:
                    cond = cond.replace("letzte_entnahme", "lagerzugang")
            resolved_conditions.append(cond)
    
        return resolved_fields, resolved_conditions

    def build_sql_prompt(self, user_query, top_k=3):
        
        query_embedding = self.embedding_model.encode(user_query, convert_to_tensor=True).tolist()
    
        # Step 0: Try matching a known wiki entry first
        matched_wiki = self.get_best_matching_wiki_entry(user_query)
        #Condition 1- if found in wiki_json
        if matched_wiki:
            print(" Wiki matched directly.")
            meta = matched_wiki["metadata"]
            original_sql = meta.get("generated_sql", "")
            print('\n ORIGINAL QUERY: ',original_sql,'\n')
            where_clause = self.extract_where_clause(original_sql)

            #  Time range patch
            time_match = re.search(r"(\d{1,3})\s*(tage|tag|days|day|wochen|woche|monate|monat)", user_query.lower())
            if time_match:
                value = int(time_match.group(1))
                unit = time_match.group(2)
                days = value * 7 if "woche" in unit else value * 30 if "monat" in unit else value
                where_clause = re.sub(
                    r"DATEADD\(DAY,\s*-\d+\s*,\s*GETDATE\(\)\)",
                    f"DATEADD(DAY, -{days}, GETDATE())",
                    where_clause,
                    flags=re.IGNORECASE
                )
                
                
            # Handle required fields
            fields = meta.get("required_fields", "").split(",")
            fields = [self.alias_mapping.get(f.strip(), f.strip()) for f in fields if f.strip()]

            has_sum = bool(re.search(r"SUM\s*\(", original_sql, re.IGNORECASE))

            # Only include fields that are not aggregate functions
            non_agg_fields = [f for f in fields if not re.search(r"(?i)\bSUM\s*\(", f)]
            
            # Patch artikelnr if found
            match = re.search(r"(?:artikel|artikelnr)[^\d]*(\d{5,})", user_query, re.IGNORECASE)
            if match:
                correct_artikel = match.group(1)
                where_clause = re.sub(
                    r"artikelnr\s*=\s*['\"]?\d+['\"]?",
                    f"artikelnr = '{correct_artikel}'",
                    where_clause,
                    flags=re.IGNORECASE
                )
                original_sql = re.sub(
                    r"artikelnr\s*=\s*['\"]?\d+['\"]?",
                    f"artikelnr = '{correct_artikel}'",
                    original_sql,
                    flags=re.IGNORECASE
                )
            
            # Patch lager
        # ---- Patch lager (robust) ----
        # extract value after the word "lager" (quoted or not), stop at quote/newline
        # 1) Try quoted value after the word "lager"
        # Patch lager only if a clear name/code is given
            lager_match = re.search(
                r"\blager\s*['\"]([A-Za-z0-9\s]+)['\"]",  # quoted value
                user_query,
                re.IGNORECASE
            )
            
            if not lager_match:
                # Fallback: lager followed by short alphanumeric code (e.g., 01 TIG), not verbs
                lager_match = re.search(
                    r"\blager\s+([A-Z]{2,}\s*\d+|\d+\s*[A-Z]{2,}|\d{2}\s+[A-Za-z]+)",
                    user_query,
                    re.IGNORECASE
                )
            
            if lager_match:
                correct_lager = lager_match.group(1).strip()
                where_clause = re.sub(
                    r"\blager\s*=\s*['\"]?.+?['\"]?",
                    f"lager = '{correct_lager}'",
                    where_clause,
                    flags=re.IGNORECASE
                )
                original_sql = re.sub(
                    r"\blager\s*=\s*['\"]?.+?['\"]?",
                    f"lager = '{correct_lager}'",
                    original_sql,
                    flags=re.IGNORECASE
                )





            # Extract TOP clause
            top_match = re.search(r"SELECT\s+TOP\s+(\d+)", original_sql, re.IGNORECASE)
            top_clause = f"TOP {top_match.group(1)}" if top_match else ""

            # Extract ORDER BY clause
            order_by_match = re.search(r"ORDER BY\s+.+", original_sql, re.IGNORECASE)
            order_by_clause = order_by_match.group(0) if order_by_match else ""

            # Extract GROUP BY clause only if present
            group_by_clause = f"GROUP BY {', '.join(non_agg_fields)}" if has_sum and non_agg_fields else ""
               
            field_str = ", ".join(sorted(set(fields)))
            
            #group_by_clause = f"\nGROUP BY {', '.join(f for f in fields if not f.upper().startswith('SUM('))}"
            #if group_by_clause:
                #print('\ngroup_by_clause is present in original query',group_by_clause,'\n')
           # else:
                #print('\ngroup_by_clause is not  present in original query\n')
            prompt = f"""
                Du bist ein SQL-Assistent. Schreibe eine SQL-Abfrage basierend auf der Benutzerfrage unten.
                
                Benutzerfrage:
                \"{user_query}\"
                
                Verwende folgende Informationen:
                - Tabelle: lagerliste
                - Gewünschte Felder: {field_str}
                - Bedingungen (WHERE): {where_clause or '1=1'}
                {"- " + top_clause if top_clause else ""}
                {"- " + order_by_clause if order_by_clause else ""}
                {"- " + group_by_clause if group_by_clause else ""}
                
                Nutze das folgende Beispiel NUR für die SQL-Syntax, nicht für die Werte:
                {original_sql}
                
                Gib NUR die finale SQL-Abfrage mit den Werten aus der Benutzerfrage zurück.

                SQL outout(Assistant): 
                
                """.strip()

            #print('/n PROMPT: /n')
            #print(prompt)
            return prompt,original_sql  
    
        # Step 1: Try RAG
        fields, template_conditions = self.resolve_relevant_templates(user_query, top_k=top_k)
        rag_has_match = bool(fields or template_conditions)
        print('rag_has_match is : ',rag_has_match)

    #Condition-2: If semantic match is found:
        concepts = build_reference_terms_de(schema)
        print("concepts: ",concepts)
        matches = extract_from_query_de(user_query, concepts)
        print('matches:',matches)
        forced_artikeltyp = None
        m = re.search(r"(?:artikel|artikelnr)[^\d]*(\d{5,})", user_query, re.IGNORECASE)
        print("semantic M: ", m)
        if m:
            print("semantic M: ", m)
            # Determine/caches packaging (your existing method)
            packaging = self.determine_and_cache_packaging(user_query)  # returns something like "Fix-Gebinde" or "Gewichtsartikel"
            # Map returned packaging to schema's artikeltypen keys (normalize defensively)
            mapping = {
                "fix": "Fix-Gebinde",
                "fix-gebinde": "Fix-Gebinde",
                "fixgebinde": "Fix-Gebinde",
                "gewichtsartikel": "Gewichtsartikel",
                "gewichtsvariable": "Gewichtsartikel",
                "gewicht": "Gewichtsartikel"
            }
            forced_artikeltyp = mapping.get(str(packaging).strip().lower(), None)
            print("forced_artikeltyp: ",forced_artikeltyp)
        if (matches.get("konzepte") and len(matches["konzepte"]) > 0):
            #print('matches:',matches)
            if forced_artikeltyp:
                resolved = resolve_hierarchical_concepts(
                            schema,
                            matches.get("konzepte", []),
                            forced_artikeltyp=forced_artikeltyp)
            else:
                resolved = resolve_hierarchical_concepts(schema, matches["konzepte"],forced_artikeltyp)
                print("*****************")
                print('resolved: ',resolved)
                print("*****************")
            database_schema =  get_database_schema_dict()
                # If the schema is a JSON string, parse it to a dictionary
            if isinstance(database_schema, str):
                try:
                    database_schema = ast.literal_eval(database_schema)
                except Exception:
                    raise ValueError("Expected database_schema to be a dict, got a malformed string.")
                #print(database_schema)
            prompt = baue_sql_prompt_aus_konzept(schema, resolved, user_query, database_schema)
                #sql_query = generate_sql_from_query(query, schema, resolved,database_schema)
            print("++++ prompt created: ")
            print(prompt)
            return prompt, None
        
    #Condition-3: If wiki data not found
        if  rag_has_match:
            
            #print("⚠️ No relevant query matched in ChromaDB — using JSON & wiki fallback.")
        
            # Step 1: Try cached or guessed artikeltyp
            artikeltyp = self.determine_and_cache_packaging(user_query)
            if not artikeltyp:
                artikeltyp = "Fix-Gebinde"  # fallback default
            print(f" Using artikeltyp: {artikeltyp}")
        
            # Step 2: Load from wiki (fields + hint)
            wiki_fields, wiki_hint = [], ""
            wiki_results = self.wiki_collection.query(query_texts=[user_query], n_results=5)
            documents = wiki_results.get("documents", [[]])[0]
            metadatas = wiki_results.get("metadatas", [[]])[0]
        
            for doc, meta in zip(documents, metadatas):
                keywords = meta.get("keywords", "").lower().split(",")
                anti_keywords = meta.get("anti_keywords", "").lower().split(",")
                query_lower = user_query.lower()
        
                if any(kw.strip() in query_lower for kw in keywords) and not any(ak.strip() in query_lower for ak in anti_keywords):
                    wiki_fields = [f.strip() for f in meta.get("required_fields", "").split(",") if f.strip()]
                    wiki_hint = meta.get("wiki_hint", "")
                    print(f" Wiki matched: fields={wiki_fields}, hint='{wiki_hint}'")
                    if not artikeltyp:
                        artikeltyp = "Fix-Gebinde" if "fix" in meta.get("keywords", "").lower() else "Gewichtsartikel"
                    break
        
            # Step 3: Merge with JSON typical fields
            typische_felder = self.config["artikeltypen"].get(artikeltyp, {}).get("typische_felder", [])
            #if not typische_felder:
                #print(f"⚠️ No typische_felder found for artikeltyp={artikeltyp}, using fallback.")
            fields = list(set((wiki_fields or []) + typische_felder))
        
            if not fields:
                # Final fallback
                fields = self.config["datenstruktur"]["lagerliste"]["primärfelder"]
        
            # Step 4: WHERE clause
            template_conditions = []
        
            base_filter = self.config.get("bestandsregeln", {}).get("filter")
            if base_filter:
                template_conditions.append(base_filter)
        
            erkennung = self.config["artikeltypen"].get(artikeltyp, {}).get("erkennbar_an", "")
            if erkennung and erkennung not in template_conditions:
                template_conditions.append(erkennung)
        
            match = re.search(r"(?:artikel|artikelnr)[^\d]*(\d{5,})", user_query.lower())
            if match:
                template_conditions.append(f"artikelnr = '{match.group(1)}'")
        
            # Step 5: Format prompt
            resolved_fields = [self.alias_mapping.get(f, f) for f in sorted(set(fields))]

            for required_field in ["artikelnr", "artikeltext1"]:
                if required_field not in resolved_fields:
                    resolved_fields.append(required_field)
                
            cleaned_conditions = self.clean_where_conditions(template_conditions, user_query)
        
            field_str = ", ".join(sorted(set(resolved_fields))) or "*"
            condition_str = " AND ".join(sorted(cleaned_conditions)) or "1=1"
        
            #group_by_fields = [f.split(" AS ")[-1] for f in resolved_fields if not f.upper().startswith("SUM(")]
            group_by_fields = [
                f.split(" AS ")[-1]
                for f in resolved_fields
                if not f.upper().startswith("SUM(") and f.strip().lower() != "rest"
            ]

            group_by_clause = f"\nGROUP BY {', '.join(sorted(set(group_by_fields)))}" if group_by_fields else ""
            
            semantic_context = self.get_semantic_context_text()
            #print('\n semantic_context')
            #print(semantic_context)
            prompt = f"""
Du bist ein SQL-Assistent. Schreibe eine SQL-Abfrage basierend auf der Benutzerfrage unten.

Benutzerfrage:
\"{user_query}\"

Semantisches Modell (zur Bedeutung der Felder):
{semantic_context}

Verwende folgende Informationen:
- Tabelle: lagerliste
- Gewünschte Felder: {field_str}
- Bedingungen (WHERE): {condition_str}
- group_by_clause: {group_by_clause}

**Verbindliche Geschäftsregeln:**  
    - Für **Fix-Gebinde-Artikel**:
        - Der Bestand **muss** über das Feld `rest` summiert werden (`SUM(rest)`).
        - Der Verbrauch **muss** über das Feld `anzahl` summiert werden (`SUM(anzahl)`).
        - `anzahl` darf **nicht** zur Summierung des Bestands verwendet werden.
        - Einheit ist `anzahleinheit`.
        - **Wenn eine Spalte aggregiert wird (z. B. `rest` → `SUM(rest)`), darf sie nicht unverändert in SELECT stehen und nicht in GROUP BY erscheinen.**
        - **Alle weiteren Felder aus der Liste der gewünschten Felder, die keine Aggregatfunktionen enthalten, müssen in der SELECT- und GROUP BY-Klausel enthalten sein.**

    - Für **gewichtsvariable Artikel**:
        - Verwende `RestGesamtgewicht` bzw. `VerbrauchGesamtgewicht`.
        - Einheiten kommen aus `GebindeGesamtgewichtGeliefert` und `anzahleinheit`.
        - **Wenn eine Spalte aggregiert wird (z. B. `RestGesamtgewicht` → `SUM(RestGesamtgewicht)`), darf sie nicht unverändert in SELECT stehen und nicht in GROUP BY erscheinen.**
        - **Alle weiteren Felder aus der Liste der gewünschten Felder, die keine Aggregatfunktionen enthalten, müssen in der SELECT- und GROUP BY-Klausel enthalten sein.**

**Example (Fix-Gebinde-Artikel):**
SELECT anzahleinheit, artikelnr, artikeltext1, SUM(rest) AS total_bestand
FROM lagerliste
WHERE lagerverbraucht IS NULL
GROUP BY anzahleinheit, artikelnr, artikeltext1;

Erstelle die vollständige SQL-Abfrage im Format des Beispiels oben.  
**Verwende ALLE in 'Gewünschte Felder' gelisteten Spalten, außer jene, die aggregiert werden.**  
Ignoriere keine Spalten. Gib nur die vollständige SQL-Abfrage zurück – ohne Kommentare oder Erläuterungen.
"""
            #print("✅ Final fallback SQL prompt:\n", prompt)
            return prompt,None
    


In [45]:
import json
import re
import os
from difflib import get_close_matches
import urllib.parse
from datetime import datetime
from decimal import Decimal
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer, util
import chromadb
from sqlalchemy import create_engine, text
import torch
# Load your JSON schema (simplified example)
base_path = os.path.abspath(os.path.dirname(__file__)) if '__file__' in globals() else os.getcwd()
print("base_path: ",base_path)
json_path = os.path.join(base_path, 'COBI_structured_semantic_model_json.json')

with open(json_path, 'r', encoding='utf-8') as f:
    schema = json.load(f)
    

base_path:  C:\Users\smoni\LLM_RAG\Develop


In [ ]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
from difflib import get_close_matches

def build_reference_terms_de(schema):
    concepts = set()
    #intents = set()

    # Add artikeltypen
    for key in schema.get("artikeltypen", {}):
        concepts.add(key.lower())

    # Add all bestandsregeln concepts (only keys with dict values)
    for key, val in schema.get("bestandsregeln", {}).items():
        if isinstance(val, dict):
            concepts.add(key.lower())
            aliases = val.get("alias", [])
            for alias in aliases:
                #print("alias bestandsregeln:  ",concepts)
                concepts.add(alias.lower())

    # Add antwortlayout → feldentypen
    feldentypen = schema.get("antwortlayout", {}).get("feldentypen", {})
    for key, val in feldentypen.items():
        concepts.add(key.lower())
        #for intent in val.get("templateIntent", []):
            #intents.add(intent.lower())

    return list(concepts)
    #, list(intents)


In [ ]:
def extract_from_query_de(query, concept_terms):
    query = query.lower()
    found_concepts = []
    found_intents = []

    for term in concept_terms:
        if term in query:
            found_concepts.append(term)
        else:
            match = get_close_matches(term, [query], n=1, cutoff=0.85)
            if match:
                found_concepts.append(term)

    #for intent in intent_terms:
        #if intent in query:
            #found_intents.append(intent)
        #else:
           # match = get_close_matches(intent, [query], n=1, cutoff=0.85)
            #if match:
                #found_intents.append(intent)

    return {
        "konzepte": list(set(found_concepts)),
        #"intents": list(set(found_intents))
    }


In [ ]:
def fetch_concept_details(schema, matched_concepts):
    details = {}

    for concept in matched_concepts:
        concept_lc = concept.lower()

        # Look in artikeltypen
        if concept_lc in [k.lower() for k in schema.get("artikeltypen", {})]:
            details[concept] = schema["artikeltypen"][[k for k in schema["artikeltypen"] if k.lower() == concept_lc][0]]

        # Look in bestandsregeln
        elif concept_lc in [k.lower() for k in schema.get("bestandsregeln", {})]:
            details[concept] = schema["bestandsregeln"][[k for k in schema["bestandsregeln"] if k.lower() == concept_lc][0]]

        # Look in antwortlayout → feldentypen
        elif concept_lc in [k.lower() for k in schema.get("antwortlayout", {}).get("feldentypen", {})]:
            details[concept] = schema["antwortlayout"]["feldentypen"][[k for k in schema["antwortlayout"]["feldentypen"] if k.lower() == concept_lc][0]]

    return details

def resolve_hierarchical_concepts_duplicate(schema, query_concepts,forced_artikeltyp=None, query_intents=None):
    resolved = {
        "artikeltyp": forced_artikeltyp,
        "base_stock_rule": None,
        "additional_rules": {
            "global_filter": {"bedingungen": [schema["bestandsregeln"]["global_filter"]]}
        },
        "antwortfelder": {},
        "raw_matches": query_concepts
    }

    # Normalize input concepts for reliable matching
    query_concepts_normalized = [c.lower().strip() for c in query_concepts]

    # Step 1: Detect artikeltyp
    for art_type in schema.get("artikeltypen", {}):
        if art_type.lower() in query_concepts_normalized:
            resolved["artikeltyp"] = art_type
            break

    # Step 2: Base stock rule
    if resolved["artikeltyp"]:
        resolved["base_stock_rule"] = schema["bestandsregeln"].get(resolved["artikeltyp"])

    # Step 3: Detect matching bestandsregeln (by key or alias)
    for concept in query_concepts_normalized:
        # Skip article type
        if resolved["artikeltyp"] and concept == resolved["artikeltyp"].lower():
            continue

        # 3A: Direct key match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            if concept == rule_key.lower():
                if rule_key not in resolved["additional_rules"]:
                    resolved["additional_rules"][rule_key] = rule_val
                    print(f" ###Direct match: {concept} → {rule_key}")
                # Do not break — allow multiple rules to match

        # 3B: Alias match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            aliases = rule_val.get("alias", [])
            for alias in aliases:
                alias_lc = alias.lower().strip()
                if alias_lc in concept or concept in alias_lc:
                    if rule_key not in resolved["additional_rules"]:
                        resolved["additional_rules"][rule_key] = rule_val
                        print(f"###Alias match: '{concept}' matched alias '{alias_lc}' → rule '{rule_key}'")

    # Step 4: Feldentyp matching (optional feature)
    for concept in query_concepts_normalized:
        for feld_key in schema.get("antwortlayout", {}).get("feldentypen", {}):
            if concept == feld_key.lower():
                resolved["antwortfelder"][feld_key] = schema["antwortlayout"]["feldentypen"][feld_key]
                print(f"\n+++ Feldentyp match: {concept} → {feld_key}")

    return resolved


In [ ]:
# -------- CORRECT SQL Fixer --------
def fix_sql_for_sqlserver_DUP(sql_query,original_sql=None):
    """
    Automatically corrects PostgreSQL-style syntax (`NULLS LAST`, `LIMIT`, `CURRENT_DATE`, `INTERVAL`) 
    and removes table aliases in UPDATE statements for SQL Server.
    """
    corrected_query = sql_query  # Keep original query for reference

    # Step 1: Fix `ORDER BY ... DESC NULLS LAST`
    #order_by_match = re.search(r"ORDER BY (.*?) DESC NULLS LAST", sql_query, re.IGNORECASE)
    #order_by_match = re.search(r"(.*)\bORDER BY\b\s+(.*?)\s+NULLS LAST", sql_query, re.IGNORECASE | re.DOTALL)
    order_by_match = re.search(
            r"ORDER BY\s+(.*?)\s*(ASC|DESC)?\s+NULLS LAST",
            sql_query,
            re.IGNORECASE | re.DOTALL
        )
    if order_by_match:
        order_by_column = order_by_match.group(1).strip()  # Extract the column used in ORDER BY
        order_expr = order_by_match.group(1).strip()
        direction = order_by_match.group(2).strip().upper() if order_by_match.group(2) else ""  # ASC or DESC or ''
        # SQL Server compatible ORDER BY with NULLS LAST simulation
        nulls_last_expr = (
            f"ORDER BY CASE WHEN {order_expr} IS NULL THEN 1 ELSE 0 END, {order_expr} {direction}".strip()
        )
        corrected_query = re.sub(
                r"ORDER BY\s+.*?\s*(ASC|DESC)?\s+NULLS LAST",
                nulls_last_expr,
                corrected_query,
                flags=re.IGNORECASE | re.DOTALL
            )
        # Ensure we extract SUM() column correctly
        sum_match = re.search(r"SUM\((.*?)\)", sql_query, re.IGNORECASE)
        sum_column = sum_match.group(1) if sum_match else order_by_column  # Use SUM(column) if found

        # Replace ORDER BY NULLS LAST logic
        corrected_query = re.sub(
            r"ORDER BY (.*?) DESC NULLS LAST",
            rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
            corrected_query,
            flags=re.IGNORECASE
        )
        #print("\n+++ Fixed `NULLS LAST` issue for SQL Server.")

        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        count_match = re.search(r"COUNT\((\w+\.\w+)\)\s+AS\s+(\w+)", sql_query, re.IGNORECASE)

        if count_match:
            count_column = count_match.group(1)  # Extract actual column inside COUNT()
            alias_name = count_match.group(2)    # Extract alias name
    
            # Step 2: Check if `SUM(alias_name)` is used in ORDER BY
            sum_alias_match = re.search(rf"SUM\(\s*{alias_name}\s*\)", sql_query, re.IGNORECASE)
    
            if sum_alias_match:
                # Replace `SUM(alias)` with `COUNT(column)`
                corrected_query = re.sub(
                    rf"SUM\(\s*{alias_name}\s*\)",
                    count_column,
                    corrected_query,
                    flags=re.IGNORECASE
                )
                print(f"\n+++ Fixed `SUM({alias_name})` by replacing it with `{count_column}` in ORDER BY.")


        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        incorrect_order_by_alias = re.search(
        r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC", 
        corrected_query, 
        re.IGNORECASE
        )

        if incorrect_order_by_alias:
            sum_column = incorrect_order_by_alias.group(2).strip()  # Extract column inside SUM()
    
            corrected_query = re.sub(
                r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC",
                rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
                corrected_query,
                flags=re.IGNORECASE
            )
            print("\n+++ Fixed incorrect `ORDER BY` alias issue for SUM().")

    # Step 2: Fix `LIMIT N` → Convert to `TOP N`
    limit_match = re.search(r"LIMIT (\d+)", corrected_query, re.IGNORECASE)

    if limit_match:
        limit_value = limit_match.group(1)  # Extracts the limit number

        # If query contains SELECT, replace with `TOP N`
        corrected_query = re.sub(
            r"SELECT ",
            f"SELECT TOP {limit_value} ",
            corrected_query,
            count=1,  # Only replace the first occurrence
            flags=re.IGNORECASE
        )

        # Remove LIMIT clause
        corrected_query = re.sub(r"LIMIT \d+", "", corrected_query, flags=re.IGNORECASE)

        print(f"\n+++ Fixed `LIMIT {limit_value}` issue for SQL Server.")

    # Step 3: Fix `CURRENT_DATE` and `INTERVAL 'X days'`
    interval_match = re.search(r"CURRENT_DATE\s*-\s*INTERVAL '(\d+) days'", corrected_query, re.IGNORECASE)

    if interval_match:
        interval_days = interval_match.group(1)  # Extracts the number of days
        corrected_query = re.sub(
            r"CURRENT_DATE\s*-\s*INTERVAL '\d+ days'",
            f"CAST(DATEADD(DAY, -{interval_days}, GETDATE()) AS DATE)",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(f"\n+++ Fixed `CURRENT_DATE - INTERVAL '{interval_days} days'` issue for SQL Server.")
    
    if "CURRENT_DATE" in corrected_query:
        corrected_query = re.sub(r"\bCURRENT_DATE\b", "CAST(GETDATE() AS DATE)", corrected_query, flags=re.IGNORECASE)
        print("\n+++ Fixed `CURRENT_DATE` issue for SQL Server.")

    # Step 4: Remove aliases from UPDATE statements
    update_match = re.search(r"UPDATE\s+(\w+)\s+(\w+)\s+SET", corrected_query, re.IGNORECASE)

    if update_match:
        table_name = update_match.group(1)  # Extract table name
        corrected_query = re.sub(
            rf"UPDATE\s+{table_name}\s+\w+\s+SET",
            f"UPDATE {table_name} SET",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(f"\n+++ Fixed alias issue in UPDATE statement for `{table_name}`.")

    if 'ilike' in corrected_query:
        corrected_query =  corrected_query.replace('ilike','like')
    
    if 'Capocollo di Montalcino' in corrected_query:
        corrected_query =  corrected_query.replace('Capocollo di Montalcino','"Capocollo di Montalcino"')
    # Return final corrected query
    if corrected_query != sql_query:
        print("\nFinal Fixed SQL Query:\n", corrected_query)
        return corrected_query
    else:
        #print("⚠️ No changes were made. Query remains unchanged.")
        return sql_query



def extract_sql_from_output(model_output: str):
    # Find all SQL blocks between ```sql and ```
    sql_blocks = re.findall(r"```sql\s*(.*?)\s*```", model_output, re.IGNORECASE | re.DOTALL)
    
    if sql_blocks:
        # Take the first block found
        return sql_blocks[0].strip()

    # Fallback: grab the first SELECT/INSERT/etc. until end of string or semicolon
    match = re.search(r"(SELECT|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER)[\s\S]+?(;|$)", model_output, re.IGNORECASE)
    if match:
        return match.group(0).strip().rstrip(';')

    return None

'''
# -------- Final Generator Integration --------
def generate_sql_from_query(user_query, schema, resolved, databaseschema):
    import re

    # Step 1: Build prompt
    prompt = baue_sql_prompt_aus_konzept(schema, resolved, user_query, databaseschema)
    print("*************************")
    print("\nPrompt created:\n")
    print(prompt)
    print("*************************")
    
    # Step 2: Generate from model
    sql_generator, _ = get_generators()
    result = sql_generator(
        prompt,
        max_new_tokens=300,
        temperature=0.0,
        do_sample=False,
        top_k=50,
        top_p=0.9
    )
  

    print("*************")
    print(result)
    print("*************")

    # Step 3: Extract generated SQL
    sql_query_result = result[0]['generated_text'].split("Assistant (SQL Output):")[-1].strip()
    
    # Try to extract the SQL block
    #match = re.search(r"(SELECT|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER)[^;]+;", sql_query, re.IGNORECASE | re.DOTALL)
    #if match:
        #raw_sql = match.group(0).replace("`", "").strip()
        #cleaned_sql = fix_sql_for_sqlserver(raw_sql)
        #print(f"🤖 Cleaned SQL Query: {cleaned_sql}")
        #return cleaned_sql
    sql_query = extract_sql_from_output(sql_query_result)

    if sql_query:
        cleaned_sql = fix_sql_for_sqlserver(sql_query)
        #print(f"🤖 Cleaned SQL Query: {cleaned_sql}")
        return cleaned_sql
    else:
        print("🤖 I could not extract a valid SQL query. Please try again.")
        #print(" Raw model output:")
        print(sql_query)
        return None
'''

def resolve_hierarchical_concepts_workingold(schema, query_concepts, forced_artikeltyp=None,query_intents=None):
    resolved = {
        "artikeltyp": forced_artikeltyp,
        "base_stock_rule": None,
        "additional_rules": {
            "global_filter": {"bedingungen": [schema["bestandsregeln"]["global_filter"]]}
        },
        "antwortfelder": {},
        "raw_matches": query_concepts
    }

    # Normalize input concepts for reliable matching
    query_concepts_normalized = [c.lower().strip() for c in query_concepts]

    # Step 1: Detect artikeltyp
    for art_type in schema.get("artikeltypen", {}):
        if art_type.lower() in query_concepts_normalized:
            resolved["artikeltyp"] = art_type
            break

    # Step 2: Base stock rule
    if resolved["artikeltyp"]:
        resolved["base_stock_rule"] = schema["bestandsregeln"].get(resolved["artikeltyp"])

    # Step 3: Detect matching bestandsregeln (by key or alias)
    for concept in query_concepts_normalized:
        # Skip article type
        if resolved["artikeltyp"] and concept == resolved["artikeltyp"].lower():
            continue

        # 3A: Direct key match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            if concept == rule_key.lower():
                if rule_key not in resolved["additional_rules"]:
                    resolved["additional_rules"][rule_key] = rule_val
                    print(f"\n+++ Direct match: {concept} → {rule_key}")
                # Do not break — allow multiple rules to match

        # 3B: Alias match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            aliases = rule_val.get("alias", [])
            for alias in aliases:
                alias_lc = alias.lower().strip()
                if alias_lc in concept or concept in alias_lc:
                    if rule_key not in resolved["additional_rules"]:
                        resolved["additional_rules"][rule_key] = rule_val
                        print(f"\n+++ Alias match: '{concept}' matched alias '{alias_lc}' → rule '{rule_key}'")

    # Step 4: Feldentyp matching (optional feature)
    for concept in query_concepts_normalized:
        for feld_key in schema.get("antwortlayout", {}).get("feldentypen", {}):
            if concept == feld_key.lower():
                resolved["antwortfelder"][feld_key] = schema["antwortlayout"]["feldentypen"][feld_key]
                print(f"\n+++ Feldentyp match: {concept} → {feld_key}")

    # Step 5: If Kritikalität is resolved but no artikeltyp was identified, infer conditions for both types
    if "Kritikalität" in resolved["additional_rules"] and not resolved["artikeltyp"]:
        def build_artikeltypen_conditions_from_json(schema):
            artikeltypen = schema.get("artikeltypen", {})
            bestandsregeln = schema.get("bestandsregeln", {})
            conditions = []
    
            for typ in artikeltypen:
                rule = bestandsregeln.get(typ, {})
                condition = rule.get("erkennbar_an") or artikeltypen[typ].get("erkennbar_an")
                if condition:
                    conditions.append(f"({condition})")
    
            return " OR ".join(conditions) if conditions else None
    
        resolved["extra_artikeltypen_condition"] = build_artikeltypen_conditions_from_json(schema)

    return resolved


In [ ]:
def resolve_hierarchical_concepts(schema, query_concepts, forced_artikeltyp=None, query_intents=None):
    """
    If `forced_artikeltyp` is provided, it is treated as the resolved artikeltyp
    (case-insensitive match to schema keys). If not provided or invalid, the type
    is inferred from `query_concepts` as before.
    """
    # Normalize input concepts for reliable matching
    query_concepts_normalized = [c.lower().strip() for c in (query_concepts or [])]

    # Canonicalize a possibly forced artikeltyp against the schema keys
    def _canonicalize_artikeltyp(value):
        if not value:
            return None
        val_lc = value.lower().strip()
        for key in schema.get("artikeltypen", {}).keys():
            if key.lower().strip() == val_lc:
                return key  # return canonical key from schema
        return None  # unknown value

    # Initialize resolved structure
    resolved = {
        "artikeltyp": None,
        "base_stock_rule": None,
        "additional_rules": {
            "global_filter": {"bedingungen": [schema["bestandsregeln"]["global_filter"]]}
        },
        "antwortfelder": {},
        "raw_matches": query_concepts
    }

    # Step 0: Respect forced_artikeltyp if valid; otherwise fall back to detection
    forced_canonical = _canonicalize_artikeltyp(forced_artikeltyp)
    if forced_canonical:
        resolved["artikeltyp"] = forced_canonical
    else:
        # Step 1: Detect artikeltyp from query concepts (only if not forced)
        for art_type in schema.get("artikeltypen", {}):
            if art_type.lower() in query_concepts_normalized:
                resolved["artikeltyp"] = art_type
                break

    # Step 2: Base stock rule (works for either forced or detected artikeltyp)
    if resolved["artikeltyp"]:
        resolved["base_stock_rule"] = schema.get("bestandsregeln", {}).get(resolved["artikeltyp"])

    # Step 3: Detect matching bestandsregeln (by key or alias)
    for concept in query_concepts_normalized:
        # Skip exact article type token
        if resolved["artikeltyp"] and concept == resolved["artikeltyp"].lower():
            continue

        # 3A: Direct key match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            if concept == rule_key.lower():
                if rule_key not in resolved["additional_rules"]:
                    resolved["additional_rules"][rule_key] = rule_val
                    # print(f"\n+++ Direct match: {concept} → {rule_key}")

        # 3B: Alias match
        for rule_key, rule_val in schema.get("bestandsregeln", {}).items():
            if not isinstance(rule_val, dict):
                continue
            for alias in rule_val.get("alias", []):
                alias_lc = alias.lower().strip()
                if alias_lc in concept or concept in alias_lc:
                    if rule_key not in resolved["additional_rules"]:
                        resolved["additional_rules"][rule_key] = rule_val
                        # print(f"\n+++ Alias match: '{concept}' matched alias '{alias_lc}' → rule '{rule_key}'")

    # Step 4: Feldentyp matching (optional feature)
    for concept in query_concepts_normalized:
        for feld_key in schema.get("antwortlayout", {}).get("feldentypen", {}):
            if concept == feld_key.lower():
                resolved["antwortfelder"][feld_key] = schema["antwortlayout"]["feldentypen"][feld_key]
                # print(f"\n+++ Feldentyp match: {concept} → {feld_key}")

    # Step 5: If Kritikalität is resolved but no artikeltyp was identified, infer OR conditions across types
    if "Kritikalität" in resolved["additional_rules"] and not resolved["artikeltyp"]:
        def build_artikeltypen_conditions_from_json(schema_):
            artikeltypen = schema_.get("artikeltypen", {})
            bestandsregeln = schema_.get("bestandsregeln", {})
            conditions = []
            for typ in artikeltypen:
                rule = bestandsregeln.get(typ, {})
                condition = rule.get("erkennbar_an") or artikeltypen[typ].get("erkennbar_an")
                if condition:
                    conditions.append(f"({condition})")
            return " OR ".join(conditions) if conditions else None

        resolved["extra_artikeltypen_condition"] = build_artikeltypen_conditions_from_json(schema)

    return resolved


In [42]:
'''def baue_sql_prompt_aus_konzept(schema, resolved, benutzerfrage,database_schema):
    import re
    threshold_match = re.search(r"bestandsfuehrungRLZvk\s*=?\s*(\d+)", benutzerfrage)
    critical_threshold = threshold_match.group(1) if threshold_match else None

    prompt = [f"Du bist ein SQL-Experte. Erstelle eine präzise SQL-Abfrage basierend auf der folgenden Benutzerfrage und den gegebenen Lagerregeln.\n"]
    prompt.append(f"### 📌 Benutzerfrage:\n{benutzerfrage}\n")

    if resolved.get("artikeltyp"):
        prompt.append(f"### 🧩 Artikeltyp erkannt: {resolved['artikeltyp']}\n")

    if resolved.get("base_stock_rule"):
        base = resolved["base_stock_rule"]
        prompt.append("### 🗂️ Basis-Bestandsregel (aus Artikeltyp):")
        prompt.append(f"- Formel: {base.get('formel')}")
        prompt.append(f"- Einheit: {base.get('einheit')}")
        prompt.append(f"- Relevante Felder: {', '.join(base.get('felder', []))}")
        if base.get("erkennbar_an"):
            prompt.append(f"- Erkennungsbedingung: {base.get('erkennbar_an')}")
        prompt.append("")

    # 🔄 Extra artikeltypen logic (when no specific artikeltyp is given, but multiple are inferred)
    if not resolved.get("artikeltyp") and resolved.get("extra_artikeltypen_condition"):
        artikel_type=resolved['artikeltyp']
        prompt.append("\n ++++Inside Extra artikeltypen logic: check artikeltype in resolved: {artikel_type}")
        prompt.append("### ➕ Zusätzliche Regel: Artikeltyp-Erkennungsbedingung (automatisch)")
        prompt.append("- Bedingungen: " + resolved["extra_artikeltypen_condition"])
    
        # Include relevant fields
        artikeltypen = schema.get("artikeltypen", {})
        bestandsregeln = schema.get("bestandsregeln", {})
    
        for typ in artikeltypen:
            regel = bestandsregeln.get(typ, {})
            felder = regel.get("felder", artikeltypen[typ].get("felder", []))
            if felder:
                prompt.append(f"- Felder für '{typ}': {', '.join(felder)}")
        prompt.append("")


    kritikal_flag= False
    for regel, details in resolved.get("additional_rules", {}).items():
        if  "Kritikalität" in regel:
            kritikal_flag=True
        if not isinstance(details, dict):
            continue
        prompt.append(f"### ➕ Zusätzliche Regel: {regel}")
        if "beschreibung" in details:
            prompt.append(f"- Beschreibung: {details['beschreibung']}")
        if "felder" in details:
            prompt.append(f"- Felder: {', '.join(details['felder'])}")
        if "bedingungen" in details:
            replaced_bedingungen = []
            for cond in details['bedingungen']:
                if critical_threshold:
                    cond = cond.replace("{critical_threshold}", critical_threshold)
                replaced_bedingungen.append(cond)
            prompt.append(f"- Bedingungen: {' UND '.join(replaced_bedingungen)}")
        prompt.append("")

    # Explicit schema and rules
    prompt.append("### 🧾 Schema von `lagerliste` (auszugsweise):")
    # Tabellenstruktur
    prompt.append("-- Tabellenstruktur:")
    if isinstance(schema, dict):
        for table, columns in database_schema.items():
            prompt.append(f"-- {table} Table")
            for col in columns:
                colname = col.get("column_name", "UNKNOWN")
                dtype = col.get("data_type", "UNKNOWN")
                prompt.append(f"--   - {colname} ({dtype})")
    else:
        prompt.append("--   (Warnung: Unerwartetes Schemaformat)")
        prompt.append(f"--   Raw: {schema}")
    prompt.append("")


    threshold_val = critical_threshold or "10"  # fallback if not found in user query
    if kritikal_flag: 
        prompt.append("### 📎 Beispiel-1 (FIX-gebinde) für korrektes SQL:")
        prompt.append(f"""SELECT artikelnr, artikeltext1, anzahleinheit, SUM(rest) AS total_rest,bestandsfuehrungRLZvk,bestandsfuehrungRLZuew,qskritisch,gesperrtKennzeichen
                    FROM lagerliste
                    WHERE colligeliefert IS NOT NULL
                      AND lagerverbraucht IS NULL
                      AND (bestandsfuehrungRLZvk <= {threshold_val} OR bestandsfuehrungRLZuew IS NOT NULL OR qskritisch = '1' OR gesperrtKennzeichen = '1')
                    GROUP BY artikelnr, artikeltext1, anzahleinheit,bestandsfuehrungRLZvk,bestandsfuehrungRLZuew,qskritisch,gesperrtKennzeichen
            """)
        prompt.append("### 📎 Beispiel-2(Gesamtgewicht) für korrektes SQL:")
        prompt.append(f""" 
            SELECT artikelnr, artikeltext1, SUM(rest) AS total_rest,  SUM(RestGesamtgewicht) as total_RestGesamtgewicht, GebindeGesamtgewichtGeliefert, anzahleinheit,bestandsfuehrungRLZvk,bestandsfuehrungRLZuew,qskritisch,gesperrtKennzeichen
            FROM lagerliste
            WHERE ColliGesamtgewicht IS NOT NULL
              AND lagerverbraucht IS NULL
              AND (bestandsfuehrungRLZvk <= {threshold_val} OR bestandsfuehrungRLZuew IS NOT NULL OR qskritisch = '1' OR gesperrtKennzeichen = '1')
            GROUP BY artikelnr, artikeltext1, GebindeGesamtgewichtGeliefert,anzahleinheit,bestandsfuehrungRLZvk,bestandsfuehrungRLZuew,qskritisch,gesperrtKennzeichen
        """)
        if not resolved.get("artikeltyp") and resolved.get("extra_artikeltypen_condition"):
                # If specific type is given, show only the relevant example
                #if resolved["artikeltyp"] == "Fix-Gebinde":
                    #prompt.append("### 📎 Beispiel: Fix-Gebinde")
                   # prompt.append(f"""SELECT ... colligeliefert IS NOT NULL ...""")
                #elif resolved["artikeltyp"] == "Gewichtsartikel":
                    #prompt.append("### 📎 Beispiel: Gewichtsartikel")
                   #prompt.append(f"""SELECT ... ColliGesamtgewicht IS NOT NULL ...""")
            #else:
                # COMBINED SQL when artikeltyp is NOT given
                prompt.append("### 📎 Beispiel: Kombinierte Artikeltypen")
                prompt.append(f"""
            SELECT artikelnr, artikeltext1, anzahleinheit,
                   SUM(rest) AS total_rest,
                   SUM(CASE WHEN ColliGesamtgewicht IS NOT NULL THEN RestGesamtgewicht ELSE 0 END) AS total_RestGesamtgewicht,
                   GebindeGesamtgewichtGeliefert,
                   bestandsfuehrungRLZvk,
                   bestandsfuehrungRLZuew,
                   qskritisch,
                   gesperrtKennzeichen
            FROM lagerliste
            WHERE (
                    (colligeliefert IS NOT NULL AND lagerverbraucht IS NULL)
                    OR (ColliGesamtgewicht IS NOT NULL)
                  )
              AND lagerverbraucht IS NULL
              AND (bestandsfuehrungRLZvk <= {threshold_val}
                   OR bestandsfuehrungRLZuew IS NOT NULL
                   OR qskritisch = '1'
                   OR gesperrtKennzeichen = '1')
            GROUP BY artikelnr, artikeltext1, anzahleinheit, GebindeGesamtgewichtGeliefert,
                     bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
            """)

    prompt.append(
        "Da in der Benutzerfrage kein spezifischer Artikeltyp genannt wurde, "
        "verwende genau die obige kombinierte Abfrage-Struktur, "
        "passe nur {critical_threshold} und Tabellen-/Spaltennamen an. "
        "Übernehme alle Felder, Aggregationen und Bedingungen exakt."
    )

    prompt.append("### 🧠 Aufgabe:\nErstelle basierend auf den obigen Regeln eine SQL-Abfrage, die die Benutzerfrage beantwortet. Verwende SQL Server Syntax.")
    prompt.append("""- Verwende **alle** Bedingungen aus den Regeln.""")
    prompt.append("""Nutze mindestens die Spalten ´artikelnr` und `artikeltext1´; gebe zusätzlich alle weiteren durch die Regeln geforderten Felder und Aggregationen aus.""")
    prompt.append("\n")
    prompt.append("""Assistant (SQL Output):""")

    return "\n".join(prompt)
'''

In [47]:
def baue_sql_prompt_aus_konzept(schema, resolved, benutzerfrage, database_schema):
    """
    Build a strict SQL-generation prompt from rules + user question.
    Adds a dynamic Output-Contract so the model must include all fields, aggregations, and conditions.
    """
    import re

    # Extract optional critical threshold from user question
    threshold_match = re.search(r"bestandsfuehrungRLZvk\s*=?\s*(\d+)", benutzerfrage or "")
    critical_threshold = threshold_match.group(1) if threshold_match else None
    threshold_val = critical_threshold or "10"  # fallback if not found

    prompt = []
    prompt.append("Du bist ein SQL-Experte. Erstelle eine präzise SQL-Abfrage basierend auf der folgenden Benutzerfrage und den gegebenen Lagerregeln.\n")
    prompt.append(f"### 📌 Benutzerfrage:\n{benutzerfrage}\n")

    # Artikeltyp (optional)
    if resolved.get("artikeltyp"):
        prompt.append(f"### 🧩 Artikeltyp erkannt: {resolved['artikeltyp']}\n")

    # Base stock rule
    if resolved.get("base_stock_rule"):
        base = resolved["base_stock_rule"]
        prompt.append("### 🗂️ Basis-Bestandsregel (aus Artikeltyp):")
        prompt.append(f"- Formel: {base.get('formel')}")
        prompt.append(f"- Einheit: {base.get('einheit')}")
        prompt.append(f"- Relevante Felder: {', '.join(base.get('felder', []))}")
        if base.get("erkennbar_an"):
            prompt.append(f"- Erkennungsbedingung: {base.get('erkennbar_an')}")
        prompt.append("")

    # Extra artikeltypen logic (only when no specific artikeltyp is given, but multiple are inferred)
    if not resolved.get("artikeltyp") and resolved.get("extra_artikeltypen_condition"):
        prompt.append("\n### ➕ Zusätzliche Regel: Artikeltyp-Erkennungsbedingung (automatisch)")
        prompt.append("- Bedingungen: " + resolved["extra_artikeltypen_condition"])
        # Include relevant fields for each known type
        artikeltypen = (schema or {}).get("artikeltypen", {})
        bestandsregeln = (schema or {}).get("bestandsregeln", {})
        for typ in artikeltypen:
            regel = bestandsregeln.get(typ, {})
            felder = regel.get("felder", artikeltypen[typ].get("felder", []))
            if felder:
                prompt.append(f"- Felder für '{typ}': {', '.join(felder)}")
        prompt.append("")

    # Additional rules
    kritikal_flag = False
    for regel, details in (resolved.get("additional_rules") or {}).items():
        if "Kritikalität" in (regel or ""):
            kritikal_flag = True
        if not isinstance(details, dict):
            continue
        prompt.append(f"### ➕ Zusätzliche Regel: {regel}")
        if "beschreibung" in details:
            prompt.append(f"- Beschreibung: {details['beschreibung']}")
        if "felder" in details:
            prompt.append(f"- Felder: {', '.join(details['felder'])}")
        if "bedingungen" in details:
            replaced_bedingungen = []
            for cond in details['bedingungen']:
                c = cond
                if critical_threshold:
                    c = c.replace("{critical_threshold}", critical_threshold)
                # Use SQL AND in prompt text to keep model in SQL mode
                c = c.replace(" UND ", " AND ")
                replaced_bedingungen.append(c)
            prompt.append(f"- Bedingungen: {' AND '.join(replaced_bedingungen)}")
        prompt.append("")

    # Explicit schema (comments only)
    prompt.append("### 🧾 Schema von `lagerliste` (auszugsweise):")
    prompt.append("-- Tabellenstruktur:")
    if isinstance(database_schema, dict):
        for table, columns in database_schema.items():
            prompt.append(f"-- {table} Table")
            for col in columns:
                colname = col.get("column_name", "UNKNOWN")
                dtype = col.get("data_type", "UNKNOWN")
                prompt.append(f"--   - {colname} ({dtype})")
    else:
        prompt.append("--   (Warnung: Unerwartetes Schemaformat)")
        prompt.append(f"--   Raw: {schema}")
    prompt.append("")

    # Optional examples when "Kritikalität" present
    if kritikal_flag:
        prompt.append("### 📎 Beispiel-1 (FIX-gebinde) für korrektes SQL:")
        prompt.append(f"""SELECT artikelnr, artikeltext1, anzahleinheit, SUM(rest) AS total_rest, bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
FROM lagerliste
WHERE colligeliefert IS NOT NULL
  AND lagerverbraucht IS NULL
  AND (bestandsfuehrungRLZvk <= {threshold_val} OR bestandsfuehrungRLZuew IS NOT NULL OR qskritisch = '1' OR gesperrtKennzeichen = '1')
GROUP BY artikelnr, artikeltext1, anzahleinheit, bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
""")
        prompt.append("### 📎 Beispiel-2 (Gesamtgewicht) für korrektes SQL:")
        prompt.append(f"""SELECT artikelnr, artikeltext1, SUM(rest) AS total_rest, SUM(RestGesamtgewicht) AS total_RestGesamtgewicht, GebindeGesamtgewichtGeliefert, anzahleinheit, bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
FROM lagerliste
WHERE ColliGesamtgewicht IS NOT NULL
  AND lagerverbraucht IS NULL
  AND (bestandsfuehrungRLZvk <= {threshold_val} OR bestandsfuehrungRLZuew IS NOT NULL OR qskritisch = '1' OR gesperrtKennzeichen = '1')
GROUP BY artikelnr, artikeltext1, GebindeGesamtgewichtGeliefert, anzahleinheit, bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
""")
        if not resolved.get("artikeltyp") and resolved.get("extra_artikeltypen_condition"):
            prompt.append("### 📎 Beispiel: Kombinierte Artikeltypen")
            prompt.append(f"""SELECT artikelnr, artikeltext1, anzahleinheit,
       SUM(rest) AS total_rest,
       SUM(CASE WHEN ColliGesamtgewicht IS NOT NULL THEN RestGesamtgewicht ELSE 0 END) AS total_RestGesamtgewicht,
       GebindeGesamtgewichtGeliefert,
       bestandsfuehrungRLZvk,
       bestandsfuehrungRLZuew,
       qskritisch,
       gesperrtKennzeichen
FROM lagerliste
WHERE (
        (colligeliefert IS NOT NULL AND lagerverbraucht IS NULL)
        OR (ColliGesamtgewicht IS NOT NULL)
      )
  AND lagerverbraucht IS NULL
  AND (bestandsfuehrungRLZvk <= {threshold_val}
       OR bestandsfuehrungRLZuew IS NOT NULL
       OR qskritisch = '1'
       OR gesperrtKennzeichen = '1')
GROUP BY artikelnr, artikeltext1, anzahleinheit, GebindeGesamtgewichtGeliefert,
         bestandsfuehrungRLZvk, bestandsfuehrungRLZuew, qskritisch, gesperrtKennzeichen
""")

    # Guidance sentence about combined structure when no explicit type mentioned
    prompt.append(
        "Da in der Benutzerfrage kein spezifischer Artikeltyp genannt wurde, "
        "verwende genau die obige kombinierte Abfrage-Struktur, "
        "passe nur {critical_threshold} und Tabellen-/Spaltennamen an. "
        "Übernehme alle Felder, Aggregationen und Bedingungen exakt."
    )

    # ----------------- Build a strict, dynamic Output-Contract -----------------
    base = resolved.get("base_stock_rule", {}) or {}
    formel = base.get("formel", "") or ""

    # Normalize separators in formula (treat comma as 'AND' and parse all SUM() occurrences)
    # e.g., "SUM(RestGesamtgewicht) , SUM(rest)" -> ["RestGesamtgewicht", "rest"]
    agg_cols = re.findall(r"SUM\(\s*([A-Za-z0-9_]+)\s*\)", formel)
    agg_cols = [c for c in agg_cols if c]  # safety

    # Aggregations with stable aliases
    agg_select = [f"SUM(l.{c}) AS sum_{c}" for c in agg_cols]

    # Unit/output fields (split by '+')
    unit_fields = [u.strip() for u in (base.get("einheit") or "").split("+") if u.strip()]
    unit_select = [f"l.{u}" for u in unit_fields]

    # Additional rule fields (exclude ones already aggregated)
    extra_select = []
    for _, details in (resolved.get("additional_rules") or {}).items():
        for f in details.get("felder", []) or []:
            if f not in agg_cols:
                extra_select.append(f"l.{f}")

    # Required SELECT: always include artikelnr + artikeltext1
    required_select = ["l.artikelnr", "l.artikeltext1"] + agg_select + unit_select + extra_select

    # Deduplicate while preserving order
    seen = set()
    required_select = [c for c in required_select if not (c in seen or seen.add(c))]

    # WHERE parts from rules
    where_parts = []

    # Base recognition condition, e.g., "ColliGesamtgewicht IS NOT NULL"
    if base.get("erkennbar_an"):
        # Convert " UND " to " AND " to keep SQL style
        cond = str(base["erkennbar_an"]).replace(" UND ", " AND ")
        where_parts.append(cond)

    # Additional rules' conditions
    for _, details in (resolved.get("additional_rules") or {}).items():
        for cond in details.get("bedingungen", []) or []:
            c = cond.replace("{critical_threshold}", threshold_val).replace(" UND ", " AND ")
            where_parts.append(c)

    # Artikelnummer from the user question (5+ digits)
    art_match = re.search(r"\b(\d{5,})\b", benutzerfrage or "")
    if art_match:
        where_parts.append(f"l.artikelnr = {art_match.group(1)}")

    # GROUP BY = all non-aggregated SELECT items (i.e., those starting with "l.")
    required_group_by = [c for c in required_select if c.strip().lower().startswith("l.")]

    # Output-Contract section (non-negotiable)
    prompt.append("### ✅ Output-Contract (nicht verhandelbar):")
    prompt.append("- Return ONLY one SQL Server query. If any required item is missing, the answer is INVALID.")
    prompt.append("- Use table alias: dbo.lagerliste AS l")
    prompt.append(f"- REQUIRED SELECT COLUMNS (ALL): {', '.join(required_select)}")
    prompt.append(f"- REQUIRED WHERE (ALL): {' AND '.join(where_parts) if where_parts else '(none)'}")
    prompt.append(f"- REQUIRED GROUP BY: {', '.join(required_group_by) if required_group_by else '(none)'}")
    prompt.append("- Do NOT omit fields even if they look redundant. Do NOT add unspecified columns.")
    prompt.append("")

    # Final task instructions
    prompt.append("### 🧠 Aufgabe:\nErstelle basierend auf den obigen Regeln eine SQL-Abfrage, die die Benutzerfrage beantwortet. Verwende SQL Server Syntax.")
    prompt.append("- Verwende **alle** Bedingungen aus den Regeln.")
    prompt.append("Nutze mindestens die Spalten ´artikelnr` und `artikeltext1´; gebe zusätzlich alle weiteren durch die Regeln geforderten Felder und Aggregationen aus.")
    prompt.append("\n")
    prompt.append("Assistant (SQL Output):")

    return "\n".join(prompt)


In [48]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from torch import cuda, multiprocessing

_sql_generator = None
_response_generator = None

def get_generators():
    global _sql_generator, _response_generator
    if _sql_generator is not None and _response_generator is not None:
        return _sql_generator, _response_generator

    print("🚀 Loading SQL model...")
    model_name = "defog/sqlcoder-7b-2"
    #model_name="codellama/CodeLlama-7b-hf"
    #model_name="openlm-research/open_llama_3b_v2"
    device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'
    print("CUDA:", cuda.is_available())

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        quantization_config=quantization_config
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model.eval()

    _sql_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
    _response_generator = _sql_generator  # Same model for now

    return _sql_generator, _response_generator


In [ ]:
import re
import urllib.parse
from sqlalchemy import create_engine, text


# Step 1: Load the SQL generation model
sql_generator, _ = get_generators()

# Step 2: Function to get schema from SQL Server
def get_database_schema():
    conn_str = "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=localhost;"
        "DATABASE=topwws;"
        "UID=sa;"
        "PWD=usersystemadmin!;"
        "TrustServerCertificate=yes"
    )
    engine = create_engine(conn_str)
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_NAME IN ('lagerliste')
        """))
        schema = ""
        for row in result:
            schema += f"{row.TABLE_NAME}: {row.COLUMN_NAME} ({row.DATA_TYPE})\n"
            #print('Schema : ',schema)
        return schema
        
def get_database_schema_dict():
    conn_str = "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=localhost;"
        "DATABASE=topwws;"
        "UID=sa;"
        "PWD=usersystemadmin!;"
        "TrustServerCertificate=yes"
    )
    engine = create_engine(conn_str)
    schema = {}
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_NAME IN ('lagerliste')
        """))
        for row in result:
            table = row.TABLE_NAME
            if table not in schema:
                schema[table] = []
            schema[table].append({
                "column_name": row.COLUMN_NAME,
                "data_type": row.DATA_TYPE
            })
    return schema

# Step 3: Generate the final model prompt using RAG + schema
def build_model_prompt(user_query):
    schema = get_database_schema()
    rag_generator = SQLRAGPromptGenerator.get_instance()
    instruction, original_sql = rag_generator.build_sql_prompt(user_query)

    prompt = f"""
You are an expert in SQL Server.
Translate the instruction below into a valid and optimized SQL query.

### Schema:
{schema}

### Instruction:
{instruction}

SQL Output:
"""
    return prompt, original_sql

In [ ]:
def fix_sql_for_sqlserver_old(sql_query,original_sql=None):
    """
    Automatically corrects PostgreSQL-style syntax (`NULLS LAST`, `LIMIT`, `CURRENT_DATE`, `INTERVAL`) 
    and removes table aliases in UPDATE statements for SQL Server.
    """
    corrected_query = sql_query  # Keep original query for reference

    # Step 1: Fix `ORDER BY ... DESC NULLS LAST`
    order_by_match = re.search(r"ORDER BY (.*?) DESC NULLS LAST", sql_query, re.IGNORECASE)

    if order_by_match:
        order_by_column = order_by_match.group(1).strip()  # Extract the column used in ORDER BY

        # Ensure we extract SUM() column correctly
        sum_match = re.search(r"SUM\((.*?)\)", sql_query, re.IGNORECASE)
        sum_column = sum_match.group(1) if sum_match else order_by_column  # Use SUM(column) if found

        # Replace ORDER BY NULLS LAST logic
        corrected_query = re.sub(
            r"ORDER BY (.*?) DESC NULLS LAST",
            rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(" Fixed `NULLS LAST` issue for SQL Server.")

        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        count_match = re.search(r"COUNT\((\w+\.\w+)\)\s+AS\s+(\w+)", sql_query, re.IGNORECASE)

        if count_match:
            count_column = count_match.group(1)  # Extract actual column inside COUNT()
            alias_name = count_match.group(2)    # Extract alias name
    
            # Step 2: Check if `SUM(alias_name)` is used in ORDER BY
            sum_alias_match = re.search(rf"SUM\(\s*{alias_name}\s*\)", sql_query, re.IGNORECASE)
    
            if sum_alias_match:
                # Replace `SUM(alias)` with `COUNT(column)`
                corrected_query = re.sub(
                    rf"SUM\(\s*{alias_name}\s*\)",
                    count_column,
                    corrected_query,
                    flags=re.IGNORECASE
                )
                print(f" Fixed `SUM({alias_name})` by replacing it with `{count_column}` in ORDER BY.")


        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        incorrect_order_by_alias = re.search(
        r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC", 
        corrected_query, 
        re.IGNORECASE
        )

        if incorrect_order_by_alias:
            sum_column = incorrect_order_by_alias.group(2).strip()  # Extract column inside SUM()
    
            corrected_query = re.sub(
                r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC",
                rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
                corrected_query,
                flags=re.IGNORECASE
            )
            print(" Fixed incorrect `ORDER BY` alias issue for SUM().")

    # Step 2: Fix `LIMIT N` → Convert to `TOP N`
    limit_match = re.search(r"LIMIT (\d+)", corrected_query, re.IGNORECASE)

    if limit_match:
        limit_value = limit_match.group(1)  # Extracts the limit number

        # If query contains SELECT, replace with `TOP N`
        corrected_query = re.sub(
            r"SELECT ",
            f"SELECT TOP {limit_value} ",
            corrected_query,
            count=1,  # Only replace the first occurrence
            flags=re.IGNORECASE
        )

        # Remove LIMIT clause
        corrected_query = re.sub(r"LIMIT \d+", "", corrected_query, flags=re.IGNORECASE)

        print(f" Fixed `LIMIT {limit_value}` issue for SQL Server.")

    # Step 3: Fix `CURRENT_DATE` and `INTERVAL 'X days'`
    interval_match = re.search(r"CURRENT_DATE\s*-\s*INTERVAL '(\d+) days'", corrected_query, re.IGNORECASE)

    if interval_match:
        interval_days = interval_match.group(1)  # Extracts the number of days
        corrected_query = re.sub(
            r"CURRENT_DATE\s*-\s*INTERVAL '\d+ days'",
            f"CAST(DATEADD(DAY, -{interval_days}, GETDATE()) AS DATE)",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(f" Fixed `CURRENT_DATE - INTERVAL '{interval_days} days'` issue for SQL Server.")
    
    if "CURRENT_DATE" in corrected_query:
        corrected_query = re.sub(r"\bCURRENT_DATE\b", "CAST(GETDATE() AS DATE)", corrected_query, flags=re.IGNORECASE)
        print(" Fixed `CURRENT_DATE` issue for SQL Server.")
    #  Remove table prefixes only if one table is used
    # Extract all table names using FROM and JOIN clauses
    tables = re.findall(r"\b(?:FROM|JOIN)\s+([a-zA-Z_][\w]*)", corrected_query, re.IGNORECASE)
    
    unique_tables = set(tables)
    if len(unique_tables) == 1:
        single_table = unique_tables.pop()
        corrected_query = re.sub(rf"\b{single_table}\.", "", corrected_query, flags=re.IGNORECASE)
        #print(f" Removed table prefixes for single table: {single_table}")
    #else:
        #print(f"⚠️ Multiple tables detected ({unique_tables}) — skipping prefix removal.")
        

    #if "ORDER BY" in corrected_query :
        #corrected_query = re.sub(r"ORDER BY\s+[\w\.\s,]+(NULLS LAST|ASC|DESC)?\s*;?", "", corrected_query, flags=re.IGNORECASE)
        #print("⚠️ Removed unnecessary ORDER BY clause not asked for by user.")


    if "ORDER BY" in corrected_query and (not original_sql or "ORDER BY" not in original_sql):
        corrected_query = re.sub(
            r"ORDER BY\s+[\w\.\s,]+(NULLS LAST|ASC|DESC)?\s*;?",
            "",
            corrected_query,
            flags=re.IGNORECASE
        )
        #print("⚠️ Removed unnecessary ORDER BY clause not present in original query.")

    
    # Step 4: Remove aliases from UPDATE statements
    update_match = re.search(r"UPDATE\s+(\w+)\s+(\w+)\s+SET", corrected_query, re.IGNORECASE)

    if update_match:
        table_name = update_match.group(1)  # Extract table name
        corrected_query = re.sub(
            rf"UPDATE\s+{table_name}\s+\w+\s+SET",
            f"UPDATE {table_name} SET",
            corrected_query,
            flags=re.IGNORECASE
        )
        #print(f" Fixed alias issue in UPDATE statement for `{table_name}`.")

    if 'ilike' in corrected_query:
        corrected_query =  corrected_query.replace('ilike','like')
    
    if 'Capocollo di Montalcino' in corrected_query:
        corrected_query =  corrected_query.replace('Capocollo di Montalcino','"Capocollo di Montalcino"')
    # Return final corrected query
    if corrected_query != sql_query:
        #print("🚀 Final Fixed SQL Query:\n", corrected_query)
        return corrected_query
    else:
        #print("⚠️ No changes were made. Query remains unchanged.")
        return sql_query


In [ ]:
def fix_sql_for_sqlserver(sql_query,original_sql=None):
    """
    Automatically corrects PostgreSQL-style syntax (`NULLS LAST`, `LIMIT`, `CURRENT_DATE`, `INTERVAL`) 
    and removes table aliases in UPDATE statements for SQL Server.
    """
    corrected_query = sql_query  # Keep original query for reference

    # Step 1: Fix `ORDER BY ... DESC NULLS LAST`
    #order_by_match = re.search(r"ORDER BY (.*?) DESC NULLS LAST", sql_query, re.IGNORECASE)
    #order_by_match = re.search(r"(.*)\bORDER BY\b\s+(.*?)\s+NULLS LAST", sql_query, re.IGNORECASE | re.DOTALL)
    order_by_match = re.search(
            r"ORDER BY\s+(.*?)\s*(ASC|DESC)?\s+NULLS LAST",
            sql_query,
            re.IGNORECASE | re.DOTALL
        )
    if order_by_match:
        order_by_column = order_by_match.group(1).strip()  # Extract the column used in ORDER BY
        order_expr = order_by_match.group(1).strip()
        direction = order_by_match.group(2).strip().upper() if order_by_match.group(2) else ""  # ASC or DESC or ''
        # SQL Server compatible ORDER BY with NULLS LAST simulation
        nulls_last_expr = (
            f"ORDER BY CASE WHEN {order_expr} IS NULL THEN 1 ELSE 0 END, {order_expr} {direction}".strip()
        )
        corrected_query = re.sub(
                r"ORDER BY\s+.*?\s*(ASC|DESC)?\s+NULLS LAST",
                nulls_last_expr,
                corrected_query,
                flags=re.IGNORECASE | re.DOTALL
            )
        # Ensure we extract SUM() column correctly
        sum_match = re.search(r"SUM\((.*?)\)", sql_query, re.IGNORECASE)
        sum_column = sum_match.group(1) if sum_match else order_by_column  # Use SUM(column) if found

        # Replace ORDER BY NULLS LAST logic
        corrected_query = re.sub(
            r"ORDER BY (.*?) DESC NULLS LAST",
            rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
            corrected_query,
            flags=re.IGNORECASE
        )
        print("\n+++ Fixed `NULLS LAST` issue for SQL Server.")

        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        count_match = re.search(r"COUNT\((\w+\.\w+)\)\s+AS\s+(\w+)", sql_query, re.IGNORECASE)

        if count_match:
            count_column = count_match.group(1)  # Extract actual column inside COUNT()
            alias_name = count_match.group(2)    # Extract alias name
    
            # Step 2: Check if `SUM(alias_name)` is used in ORDER BY
            sum_alias_match = re.search(rf"SUM\(\s*{alias_name}\s*\)", sql_query, re.IGNORECASE)
    
            if sum_alias_match:
                # Replace `SUM(alias)` with `COUNT(column)`
                corrected_query = re.sub(
                    rf"SUM\(\s*{alias_name}\s*\)",
                    count_column,
                    corrected_query,
                    flags=re.IGNORECASE
                )
                print(f"\n+++ Fixed `SUM({alias_name})` by replacing it with `{count_column}` in ORDER BY.")


        # Step 2: Fix incorrect ORDER BY aliasing in SUM()
        incorrect_order_by_alias = re.search(
        r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC", 
        corrected_query, 
        re.IGNORECASE
        )

        if incorrect_order_by_alias:
            sum_column = incorrect_order_by_alias.group(2).strip()  # Extract column inside SUM()
    
            corrected_query = re.sub(
                r"ORDER BY CASE WHEN SUM\((.*?)\) IS NULL THEN 1 ELSE 0 END, SUM\((.*?)\) DESC",
                rf"ORDER BY CASE WHEN SUM({sum_column}) IS NULL THEN 1 ELSE 0 END, SUM({sum_column}) DESC",
                corrected_query,
                flags=re.IGNORECASE
            )
            print("\n+++ Fixed incorrect `ORDER BY` alias issue for SUM().")

    # Step 2: Fix `LIMIT N` → Convert to `TOP N`
    limit_match = re.search(r"LIMIT (\d+)", corrected_query, re.IGNORECASE)

    if limit_match:
        limit_value = limit_match.group(1)  # Extracts the limit number

        # If query contains SELECT, replace with `TOP N`
        corrected_query = re.sub(
            r"SELECT ",
            f"SELECT TOP {limit_value} ",
            corrected_query,
            count=1,  # Only replace the first occurrence
            flags=re.IGNORECASE
        )

        # Remove LIMIT clause
        corrected_query = re.sub(r"LIMIT \d+", "", corrected_query, flags=re.IGNORECASE)

        print(f"\n+++ Fixed `LIMIT {limit_value}` issue for SQL Server.")

    # Step 3: Fix `CURRENT_DATE` and `INTERVAL 'X days'`
    interval_match = re.search(r"CURRENT_DATE\s*-\s*INTERVAL '(\d+) days'", corrected_query, re.IGNORECASE)

    if interval_match:
        interval_days = interval_match.group(1)  # Extracts the number of days
        corrected_query = re.sub(
            r"CURRENT_DATE\s*-\s*INTERVAL '\d+ days'",
            f"CAST(DATEADD(DAY, -{interval_days}, GETDATE()) AS DATE)",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(f"\n+++ Fixed `CURRENT_DATE - INTERVAL '{interval_days} days'` issue for SQL Server.")
    
    if "CURRENT_DATE" in corrected_query:
        corrected_query = re.sub(r"\bCURRENT_DATE\b", "CAST(GETDATE() AS DATE)", corrected_query, flags=re.IGNORECASE)
        print("\n+++ Fixed `CURRENT_DATE` issue for SQL Server.")
													
	
    # Step 4: Remove aliases from UPDATE statements
    update_match = re.search(r"UPDATE\s+(\w+)\s+(\w+)\s+SET", corrected_query, re.IGNORECASE)

    if update_match:
        table_name = update_match.group(1)  # Extract table name
        corrected_query = re.sub(
            rf"UPDATE\s+{table_name}\s+\w+\s+SET",
            f"UPDATE {table_name} SET",
            corrected_query,
            flags=re.IGNORECASE
        )
        print(f"\n+++ Fixed alias issue in UPDATE statement for `{table_name}`.")

    if 'ilike' in corrected_query:
        corrected_query =  corrected_query.replace('ilike','like')
    
    if 'Capocollo di Montalcino' in corrected_query:
        corrected_query =  corrected_query.replace('Capocollo di Montalcino','"Capocollo di Montalcino"')
    # Return final corrected query
    if corrected_query != sql_query:
        print("\nFinal Fixed SQL Query:\n", corrected_query)
        return corrected_query
    else:
        #print("⚠️ No changes were made. Query remains unchanged.")
        return sql_query


In [ ]:
def generate_sql_query(user_query):
    #print('inside generate_sql_query')
    prompt,original_sql = build_model_prompt(user_query)

    # ✅ Skip model if wiki-style instruction is detected
    #if "Gib die finale SQL-Abfrage zurück" in prompt:
       # print("⚠️ Detected SQL-style instruction from wiki. Manually extracting SQL.")
    '''
        # Extract parts from prompt
        where_match = re.search(r"Bedingungen \(WHERE\):\s*(.+?)\n", prompt)
        fields_match = re.search(r"Gewünschte Felder:\s*(.+?)\n", prompt)
        table_match = re.search(r"Tabelle:\s*(\w+)", prompt)

        # Look for a line that starts with GROUP BY
        group_by_clause = ""
        for line in prompt.splitlines():
            if line.strip().startswith("GROUP BY"):
                group_by_clause = line.strip()
                break

        if where_match and fields_match and table_match:
            where_clause = where_match.group(1).strip()
            fields = fields_match.group(1).strip()
            table_name = table_match.group(1).strip()

            sql = f"SELECT {fields}\nFROM {table_name}\nWHERE {where_clause}"
            if group_by_clause:
                sql += f"\n{group_by_clause}"
            return sql.strip()

        print("❌ Failed to extract SQL components. Returning raw prompt.")
        return prompt.strip()
    '''
    # Fallback: use model to generate SQL
    result = sql_generator(prompt, max_new_tokens=300, temperature=0.0, do_sample=False)
    if not result:
        print('❌ No result generated by model')
        return ""

    raw_output = result[0]["generated_text"]

    #print("\n+++raw_output:", raw_output)
    if result[0]['generated_text'].split("Assistant (SQL Output):")[-1].strip():
         raw_output = result[0]['generated_text'].split("Assistant (SQL Output):")[-1].strip()

   #"SQL Output:"
    if result[0]['generated_text'].split("SQL Output:")[-1].strip():
         raw_output = result[0]['generated_text'].split("SQL Output:")[-1].strip()

        
    #   Remove everything before SELECT/UPDATE/etc.
    match = re.search(r"(SELECT|UPDATE|INSERT|DELETE|WITH)\s.+", raw_output, re.IGNORECASE | re.DOTALL)
    if match:
        sql_output = match.group(0).strip()
    else:
        print("⚠️ Could not detect valid SQL in model output.")
        return raw_output.strip()

    #  Fix SQL syntax issues
    sql_server_ready = fix_sql_for_sqlserver(sql_output,original_sql)
    #print("\n +++SQL sql_server_ready: ", sql_server_ready)
    
    #  Extract clean SQL statement ending with semicolon (optional)
    final_match = re.search(r"(SELECT|UPDATE|DELETE|INSERT|WITH)\s.+?;", sql_server_ready, re.IGNORECASE | re.DOTALL)

    
    return final_match.group(0).strip() if final_match else sql_server_ready.strip()

In [ ]:
# Example usage
user_question = "Welche Artikel haben einen kritischen Status?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:

# Example usage
user_question = "Welche Artikel haben eine Losnummer, aber keine Chargennummer?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
# Example usage
user_question = "Welche Artikel sind aktuell reserviert?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
user_question = "Wann wurde ein bestimmter Artikel '6854210' ins Lager aufgenommen?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [49]:
user_question = "Wie viel ist von Artikel 6852980 da (Bestand)?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


rag_has_match is :  True
concepts:  ['müssen zuerst verkauft werden', 'kritikalität', 'verkaufsgrenze erreicht', 'artikel gesperrt', 'verbotener verkauf', 'dringend verkaufen', 'qskritisch', 'rückverfolgbarkeit', 'vor verkaufsende verkaufen', 'bewertung', 'sperrung', 'muss zuerst verkauft werden', 'muss raus', 'kritisch', 'verkaufen bevor ablauf', 'verfügbarkeit', 'laufen bald ab', 'geplante_zugänge', 'fix-gebinde', 'nicht im verkauf', 'geplante_abgänge', 'mhd bald erreicht', 'produktrisiko', 'dringend abverkaufen', 'verkaufsbeschränkung', 'nicht freigegeben', 'ablaufrelevant', 'kritischen status', 'verfallsdatum', 'nicht mehr verkaufsfähig', 'überwachungsartikel', 'müssen raus', 'qs-sperre', 'kritische artikel', 'charge gesperrt', 'bald ablaufend', 'mindesthaltbarkeitsdatum erreicht', 'müssen verkauft werden', 'intern gesperrt', 'überwachungsliste', 'qualitätsproblem', 'wie viel ist von artikel', 'statusmanagement', 'mhd abgelaufen', 'verfallsnah', 'lagerposition', 'struktur', 'charge

In [50]:
user_question = "Wie viel Ware wurde bereits verbraucht artikel '6854210'? (Fix-Gebinde)?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Wiki matched directly.

 ORIGINAL QUERY:  SELECT artikelnr, artikeltext1, SUM(verbrauch) AS total_verbrauch, anzahleinheit FROM lagerliste WHERE artikelnr = '6854210' AND lagerverbraucht is not NULL  and GebindeGesamtgewichtGeliefert = ' ' GROUP BY artikelnr, artikeltext1, anzahleinheit; 

Generated SQL:
 SELECT SUM(verbrauch) AS total_verbrauch, anzahleinheit, artikelnr, artikeltext1 FROM lagerliste WHERE artikelnr = '6854210' AND lagerverbraucht is not NULL  and GebindeGesamtgewichtGeliefert ='' GROUP BY artikelnr, artikeltext1, anzahleinheit;


In [ ]:
user_question = "Gibt es noch Bestand von einem bestimmten Artikel '6856550' in einem bestimmten Lager '02 TIG'?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:

user_question = "Welche Artikel wurden zuletzt ins Lager aufgenommen?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
user_question = "Wie viel Gesamtgewicht eines gewichtsvariablen Artikels '6856001' wurde geliefert?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
user_question = "Welche Artikel haben eine Vor-Reservierung?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

In [ ]:
Wie viel von einem gewichtsvariablen Artikel '6856000' wurde bereits verbraucht?

In [ ]:
user_question = "Wie viel ist von Artikel 6854250 da?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
Wie viele Fix-Gebinde-Artikel und gewichtsvariable Artikel gibt es im System?
Wie viele Fix-Gebinde-Artikel sind auf Lager?
Wie viel Bestand gibt es für gewichtsvariable Artikel?
Welche Artikel haben einen kritischen Status?
Welche Artikel sind aktuell reserviert?
Wann wurde ein bestimmter Artikel '6854210' ins Lager aufgenommen?
Wie viel Ware wurde bereits verbraucht '6854210'? (Fix-Gebinde)
Wie viel von einem gewichtsvariablen Artikel '6856000' wurde bereits verbraucht?
Gibt es noch Bestand von einem bestimmten Artikel '6856050' in einem bestimmten Lager '01 TIG'?


In [ ]:
#1
user_question = "Wie viel ist von Artikel 6854250 da?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#2
user_question = "Wie viele Fix-Gebinde-Artikel und gewichtsvariable Artikel gibt es im System?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#3
user_question = "Wie viele Fix-Gebinde-Artikel sind auf Lager?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#4
user_question = "Wie viel Bestand gibt es für gewichtsvariable Artikel?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#5
user_question = "Welche Artikel haben einen kritischen Status?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#6
user_question = "Welche Artikel sind aktuell reserviert?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#7
user_question = "Wann wurde ein bestimmter Artikel '6854210' ins Lager aufgenommen?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#8
user_question = "Wie viel Ware wurde bereits verbraucht '6854210'? (Fix-Gebinde)"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#9
user_question = "Wie viel von einem gewichtsvariablen Artikel '6856000' wurde bereits verbraucht?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:
#10
user_question = "Gibt es noch Bestand von einem bestimmten Artikel '6856050' in einem bestimmten Lager '01 TIG'?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [ ]:

#11
user_question = "Welche Artikel wurden zuletzt ins Lager aufgenommen?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)


In [51]:
user_question = "Hat dieser Artikel '6854210' einen kritischen Status?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


rag_has_match is :  True
concepts:  ['müssen zuerst verkauft werden', 'kritikalität', 'verkaufsgrenze erreicht', 'artikel gesperrt', 'verbotener verkauf', 'dringend verkaufen', 'qskritisch', 'rückverfolgbarkeit', 'vor verkaufsende verkaufen', 'bewertung', 'sperrung', 'muss zuerst verkauft werden', 'muss raus', 'kritisch', 'verkaufen bevor ablauf', 'verfügbarkeit', 'laufen bald ab', 'geplante_zugänge', 'fix-gebinde', 'nicht im verkauf', 'geplante_abgänge', 'mhd bald erreicht', 'produktrisiko', 'dringend abverkaufen', 'verkaufsbeschränkung', 'nicht freigegeben', 'ablaufrelevant', 'kritischen status', 'verfallsdatum', 'nicht mehr verkaufsfähig', 'überwachungsartikel', 'müssen raus', 'qs-sperre', 'kritische artikel', 'charge gesperrt', 'bald ablaufend', 'mindesthaltbarkeitsdatum erreicht', 'müssen verkauft werden', 'intern gesperrt', 'überwachungsliste', 'qualitätsproblem', 'wie viel ist von artikel', 'statusmanagement', 'mhd abgelaufen', 'verfallsnah', 'lagerposition', 'struktur', 'charge

In [ ]:
#Wie viel physischer Bestand ist im Lager vorhanden?
#Wie viel Bestand ist real verfügbar, inkl. geplanter Lieferungen?
#Welche Artikel sind aktuell kritisch?'''

In [52]:
user_question = "Wie viel physischer Bestand ist im Lager vorhanden?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Wiki matched directly.

 ORIGINAL QUERY:  SELECT artikelnr, artikeltext1, SUM(rest) AS rest, anzahleinheit FROM lagerliste WHERE lagerverbraucht IS NULL GROUP BY artikelnr, artikeltext1, anzahleinheit; 

Generated SQL:
 SELECT SUM(rest) AS total_rest FROM lagerliste WHERE lagerverbraucht IS NULL;


In [53]:
user_question = "Wie viel physischer je Artikel Bestand ist im Lager vorhanden?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

rag_has_match is :  True
concepts:  ['müssen zuerst verkauft werden', 'kritikalität', 'verkaufsgrenze erreicht', 'artikel gesperrt', 'verbotener verkauf', 'dringend verkaufen', 'qskritisch', 'rückverfolgbarkeit', 'vor verkaufsende verkaufen', 'bewertung', 'sperrung', 'muss zuerst verkauft werden', 'muss raus', 'kritisch', 'verkaufen bevor ablauf', 'verfügbarkeit', 'laufen bald ab', 'geplante_zugänge', 'fix-gebinde', 'nicht im verkauf', 'geplante_abgänge', 'mhd bald erreicht', 'produktrisiko', 'dringend abverkaufen', 'verkaufsbeschränkung', 'nicht freigegeben', 'ablaufrelevant', 'kritischen status', 'verfallsdatum', 'nicht mehr verkaufsfähig', 'überwachungsartikel', 'müssen raus', 'qs-sperre', 'kritische artikel', 'charge gesperrt', 'bald ablaufend', 'mindesthaltbarkeitsdatum erreicht', 'müssen verkauft werden', 'intern gesperrt', 'überwachungsliste', 'qualitätsproblem', 'wie viel ist von artikel', 'statusmanagement', 'mhd abgelaufen', 'verfallsnah', 'lagerposition', 'struktur', 'charge

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Wiki matched: fields=['artikelnr', 'artikeltext1', 'rest', 'anzahleinheit'], hint='Der physische Bestand ist die aktuelle verfügbare Menge im Lager, ermittelt über das Feld 'rest'. Nur aktive Chargen werden berücksichtigt.'
Generated SQL:
 SELECT anzahleinheit, artikelnr, artikeltext1, SUM(rest) AS total_bestand FROM lagerliste WHERE lagerverbraucht IS NULL GROUP BY anzahleinheit, artikelnr, artikeltext1;


In [54]:
user_question = "Wie viel Bestand ist real verfügbar, inkl. geplanter Lieferungen?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Wiki matched directly.

 ORIGINAL QUERY:  SELECT artikelnr, artikeltext1, anzahleinheit, SUM(rest) AS rest FROM lagerliste WHERE lagerverbraucht IS NULL GROUP BY artikelnr, artikeltext1, anzahleinheit; 

Generated SQL:
 SELECT artikelnr, artikeltext1, anzahleinheit, SUM(rest) AS rest FROM lagerliste WHERE lagerverbraucht IS NULL GROUP BY artikelnr, artikeltext1, anzahleinheit;


In [55]:
user_question = "Welche Artikel sind aktuell kritisch?"
sql_query = generate_sql_query(user_question)
print("Generated SQL:\n", sql_query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Wiki matched directly.

 ORIGINAL QUERY:  SELECT artikelnr, artikeltext1, mhd, bestandsfuehrungRLZvk, qskritisch, gesperrtKennzeichen FROM lagerliste WHERE lagerverbraucht IS NULL AND (bestandsfuehrungRLZvk <= 7 OR qskritisch = '1' OR gesperrtKennzeichen = '1'); 

Generated SQL:
 SELECT artikelnr, artikeltext1, mhd, bestandsfuehrungRLZvk, qskritisch, gesperrtKennzeichen FROM lagerliste WHERE lagerverbraucht IS NULL AND (bestandsfuehrungRLZvk <= 7 OR qskritisch = '1' OR gesperrtKennzeichen = '1');
